# Scientific Colab: Friend DMLPA Transformer vs PPO MLP

This notebook is prepared for a serious Google Colab run, not for a local demo. It compares your friend's DMLPA-style Transformer feature extractor against a standard PPO MLP baseline on the thesis-faithful DKANA Gymnasium environment.

Main scientific question: **does the DMLPA/Transformer architecture improve policy performance over a strong PPO MLP baseline under the same thesis-faithful environment, action contract, reward, risk level, frame stack, training budget, and evaluation protocol?**

## Experimental contract

- Environment: `make_dkana_thesis_faithful_env`.
- Action contract: `factorized`, i.e. `MultiDiscrete([6, 6, 6, 3])` for Op3 buffer, Op5 buffer, Op9 buffer, and assembly shifts.
- Observation: `env_sdm_history_reward` with `observation_version="v5"`.
- Temporal memory surface: `VecFrameStack(n_stack=30)` for both models.
- Baseline: PPO with standard MLP policy.
- Friend model: PPO with a custom DMLPA-style feature extractor: `Linear -> GELU -> Linear -> TransformerEncoder -> last token`.
- Serious protocol: 5 train seeds, 500,000 timesteps per policy/seed, 20 evaluation episodes per train seed.
- Primary comparison: seed-paired policy means with bootstrap confidence intervals and paired tests.

The notebook also has a `debug` profile for a short smoke run. The default is `serious`.

In [ ]:
# =====================
# 0) Scientific config
# =====================

RUN_PROFILE = "serious"  # "serious" for paper-grade Colab run; "debug" for a quick smoke test.

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/scresia"

# Keep outputs in Drive for serious Colab runs. If you do not want Drive prompts, set this False.
USE_GOOGLE_DRIVE_OUTPUT = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/scresia_results/friend_dmlpa_vs_ppo_serious"
LOCAL_OUTPUT_DIR = "/content/friend_dmlpa_vs_ppo_serious"

BASE_SEED = 42
TRAIN_SEEDS_SERIOUS = [101, 202, 303, 404, 505]
EVAL_EPISODES_SERIOUS = 20
TOTAL_TIMESTEPS_SERIOUS = 500_000
MAX_STEPS_SERIOUS = 260

TRAIN_SEEDS_DEBUG = [101]
EVAL_EPISODES_DEBUG = 2
TOTAL_TIMESTEPS_DEBUG = 256
MAX_STEPS_DEBUG = 8

REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
STOCHASTIC_PT = True
OBSERVATION_VERSION = "v5"
OBSERVATION_MODE = "env_sdm_history_reward"
ACTION_SPACE_MODE = "factorized"
LEARN_INITIAL_DECISION = True
STEP_SIZE_HOURS = 168.0

FRAME_STACK = 30
FRIEND_FEATURES_DIM = 120
FRIEND_NHEAD = 12
FRIEND_TRANSFORMER_LAYERS = 4

LEARNING_RATE = 3e-4
N_STEPS = 1024
BATCH_SIZE = 256
N_EPOCHS = 10
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.20
ENT_COEF = 0.00
VF_COEF = 0.50
MAX_GRAD_NORM = 0.50

RESUME_IF_MODEL_EXISTS = True
SAVE_EVERY_TRAINED_MODEL = True

if RUN_PROFILE == "serious":
    TRAIN_SEEDS = TRAIN_SEEDS_SERIOUS
    EVAL_EPISODES = EVAL_EPISODES_SERIOUS
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_SERIOUS
    MAX_STEPS = MAX_STEPS_SERIOUS
elif RUN_PROFILE == "debug":
    TRAIN_SEEDS = TRAIN_SEEDS_DEBUG
    EVAL_EPISODES = EVAL_EPISODES_DEBUG
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_DEBUG
    MAX_STEPS = MAX_STEPS_DEBUG
    N_STEPS = 64
    BATCH_SIZE = 32
    N_EPOCHS = 2
else:
    raise ValueError("RUN_PROFILE must be 'serious' or 'debug'.")

POLICY_LABELS = ["ppo_mlp_baseline", "ppo_friend_dmlpa_transformer"]

print({
    "run_profile": RUN_PROFILE,
    "train_seeds": TRAIN_SEEDS,
    "eval_episodes_per_seed": EVAL_EPISODES,
    "total_timesteps_per_model_seed": TOTAL_TIMESTEPS,
    "max_steps": MAX_STEPS,
    "frame_stack": FRAME_STACK,
})

In [ ]:
# =====================================
# 1) Colab setup: clone repo + install
# =====================================

from __future__ import annotations

import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        if check:
            raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result


drive_mounted = False
if IN_COLAB:
    if USE_GOOGLE_DRIVE_OUTPUT:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=True, timeout_ms=120_000)
            drive_mounted = True
        except Exception as exc:
            print("WARNING: Google Drive mount failed; falling back to /content output.")
            print(type(exc).__name__, str(exc))
            USE_GOOGLE_DRIVE_OUTPUT = False

    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--branch", GIT_BRANCH, GIT_URL, REPO_DIR])
    repo_root = Path(REPO_DIR)
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_root / "requirements.txt")])
    sys.path.insert(0, "/content")
else:
    cwd = Path.cwd().resolve()
    repo_root = cwd if (cwd / "supply_chain").exists() else cwd.parent
    sys.path.insert(0, str(repo_root))

output_root = Path(DRIVE_OUTPUT_DIR if (IN_COLAB and USE_GOOGLE_DRIVE_OUTPUT and drive_mounted) else LOCAL_OUTPUT_DIR)
if not IN_COLAB:
    output_root = repo_root / "outputs" / "friend_dmlpa_vs_ppo_serious"
output_root.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("repo_root:", repo_root)
print("output_root:", output_root)
print("python:", sys.version)
print("platform:", platform.platform())

In [ ]:
# ==========================
# 2) Runtime imports + setup
# ==========================

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize

from scresia.supply_chain.external_env_interface import (
    get_episode_terminal_metrics,
    make_dkana_thesis_faithful_env,
)

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)

git_hash = run_cmd(["git", "rev-parse", "HEAD"], cwd=repo_root).stdout.strip()
print("repo git hash:", git_hash)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())

In [ ]:
# ==========================================
# 3) Save immutable run manifest/config file
# ==========================================

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_profile": RUN_PROFILE,
    "git_url": GIT_URL,
    "git_branch": GIT_BRANCH,
    "git_hash": git_hash,
    "train_seeds": TRAIN_SEEDS,
    "eval_episodes_per_seed": EVAL_EPISODES,
    "total_timesteps_per_model_seed": TOTAL_TIMESTEPS,
    "max_steps": MAX_STEPS,
    "reward_mode": REWARD_MODE,
    "risk_level": RISK_LEVEL,
    "stochastic_pt": STOCHASTIC_PT,
    "observation_version": OBSERVATION_VERSION,
    "observation_mode": OBSERVATION_MODE,
    "action_space_mode": ACTION_SPACE_MODE,
    "learn_initial_decision": LEARN_INITIAL_DECISION,
    "step_size_hours": STEP_SIZE_HOURS,
    "frame_stack": FRAME_STACK,
    "friend_features_dim": FRIEND_FEATURES_DIM,
    "friend_nhead": FRIEND_NHEAD,
    "friend_transformer_layers": FRIEND_TRANSFORMER_LAYERS,
    "ppo_hyperparameters": {
        "learning_rate": LEARNING_RATE,
        "n_steps": N_STEPS,
        "batch_size": BATCH_SIZE,
        "n_epochs": N_EPOCHS,
        "gamma": GAMMA,
        "gae_lambda": GAE_LAMBDA,
        "clip_range": CLIP_RANGE,
        "ent_coef": ENT_COEF,
        "vf_coef": VF_COEF,
        "max_grad_norm": MAX_GRAD_NORM,
    },
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
}

(output_root / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2)[:3000])

In [ ]:
# =============================================
# 4) Smoke test: environment contract is correct
# =============================================

smoke_env = make_dkana_thesis_faithful_env(
    reward_mode=REWARD_MODE,
    risk_level=RISK_LEVEL,
    stochastic_pt=STOCHASTIC_PT,
    observation_version=OBSERVATION_VERSION,
    observation_mode=OBSERVATION_MODE,
    action_space_mode=ACTION_SPACE_MODE,
    learn_initial_decision=LEARN_INITIAL_DECISION,
    step_size_hours=STEP_SIZE_HOURS,
    max_steps=3,
)

obs, info = smoke_env.reset(seed=BASE_SEED)
print("observation shape:", obs.shape)
print("observation space:", smoke_env.observation_space)
print("action space:", smoke_env.action_space)
print("first phase:", info.get("action_phase"))
assert smoke_env.action_space.nvec.tolist() == [6, 6, 6, 3]

action = smoke_env.action_space.sample()
obs, reward, terminated, truncated, info = smoke_env.step(action)
print("initial decision step:", {"reward": reward, "phase": info.get("action_phase")})

action = smoke_env.action_space.sample()
obs, reward, terminated, truncated, info = smoke_env.step(action)
print("first weekly step:", {
    "reward": reward,
    "phase": info.get("action_phase"),
    "new_delivered": info.get("new_delivered"),
    "pending_backorders": info.get("pending_backorders"),
    "ret_cd_step": info.get("ret_cd_step"),
})
smoke_env.close()

In [ ]:
# ==========================================
# 5) Shared environment factory and unwrapping
# ==========================================

ENV_KWARGS = dict(
    reward_mode=REWARD_MODE,
    risk_level=RISK_LEVEL,
    stochastic_pt=STOCHASTIC_PT,
    observation_version=OBSERVATION_VERSION,
    observation_mode=OBSERVATION_MODE,
    action_space_mode=ACTION_SPACE_MODE,
    learn_initial_decision=LEARN_INITIAL_DECISION,
    step_size_hours=STEP_SIZE_HOURS,
    max_steps=MAX_STEPS,
)


def make_single_env(seed: int):
    def _init():
        env = make_dkana_thesis_faithful_env(**ENV_KWARGS)
        env.reset(seed=seed)
        return Monitor(env)
    return _init


def make_vec_env(seed: int) -> VecNormalize:
    vec = DummyVecEnv([make_single_env(seed)])
    vec = VecFrameStack(vec, n_stack=FRAME_STACK)
    vec = VecNormalize(vec, norm_obs=True, norm_reward=True)
    return vec


def copy_vecnormalize_stats(source: VecNormalize, target: VecNormalize) -> None:
    target.obs_rms = source.obs_rms
    target.ret_rms = source.ret_rms
    target.clip_obs = source.clip_obs
    target.clip_reward = source.clip_reward
    target.gamma = source.gamma
    target.epsilon = source.epsilon
    target.training = False
    target.norm_reward = False


def get_base_env_from_vec(vec_env: VecNormalize):
    current = vec_env
    while hasattr(current, "venv"):
        current = current.venv
    env = current.envs[0]
    while hasattr(env, "env"):
        env = env.env
    return env

probe_vec = make_vec_env(BASE_SEED)
probe_obs = probe_vec.reset()
print("stacked observation shape:", probe_obs.shape)
print("vector action space:", probe_vec.action_space)
probe_vec.close()

In [ ]:
# =============================================================
# 6) Friend architecture: corrected DMLPA/Transformer extractor
# =============================================================

class FriendDMLPAExtractor(BaseFeaturesExtractor):
    """DMLPA-style Transformer extractor adapted from the original notebook.

    It preserves the intended architecture:
    Linear -> GELU -> Linear -> TransformerEncoder -> last temporal token.

    The correction is explicit shape handling. VecFrameStack flattens the stacked
    sequence, so the extractor reconstructs `(batch, FRAME_STACK, frame_dim)`.
    """

    def __init__(
        self,
        observation_space: gym.spaces.Box,
        frame_stack: int = FRAME_STACK,
        features_dim: int = FRIEND_FEATURES_DIM,
        nhead: int = FRIEND_NHEAD,
        num_layers: int = FRIEND_TRANSFORMER_LAYERS,
    ):
        super().__init__(observation_space, features_dim)
        flat_dim = int(np.prod(observation_space.shape))
        if flat_dim % frame_stack != 0:
            raise ValueError(
                f"Observation dim {flat_dim} is not divisible by frame_stack={frame_stack}."
            )
        if features_dim % nhead != 0:
            raise ValueError("features_dim must be divisible by nhead.")
        self.frame_stack = int(frame_stack)
        self.frame_dim = flat_dim // self.frame_stack
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.frame_dim, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim,
            nhead=nhead,
            dim_feedforward=features_dim * 2,
            batch_first=True,
            dropout=0.0,
            activation="gelu",
        )
        self.accumulated = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_norm = torch.nn.LayerNorm(features_dim)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        batch_size = observations.shape[0]
        x = observations.reshape(batch_size, self.frame_stack, self.frame_dim)
        x = self.latent_rw(x)
        x = self.accumulated(x)
        return self.output_norm(x[:, -1, :])

shape_test_vec = make_vec_env(BASE_SEED)
shape_obs = shape_test_vec.reset()
extractor = FriendDMLPAExtractor(shape_test_vec.observation_space)
with torch.no_grad():
    y = extractor(torch.as_tensor(shape_obs, dtype=torch.float32))
print("extractor input:", shape_obs.shape)
print("frame_dim:", extractor.frame_dim)
print("extractor output:", tuple(y.shape))
shape_test_vec.close()

In [ ]:
# ========================================
# 7) Policy definitions and train function
# ========================================

@dataclass
class TrainedPolicy:
    label: str
    train_seed: int
    model: PPO
    vec_env: VecNormalize
    model_path: Path
    norm_path: Path
    elapsed_seconds: float
    parameter_count: int


def count_trainable_parameters(model: torch.nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


BASELINE_POLICY_KWARGS = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
)

FRIEND_POLICY_KWARGS = dict(
    features_extractor_class=FriendDMLPAExtractor,
    features_extractor_kwargs=dict(
        frame_stack=FRAME_STACK,
        features_dim=FRIEND_FEATURES_DIM,
        nhead=FRIEND_NHEAD,
        num_layers=FRIEND_TRANSFORMER_LAYERS,
    ),
    net_arch=dict(pi=[128], vf=[128]),
)

POLICY_KWARGS = {
    "ppo_mlp_baseline": BASELINE_POLICY_KWARGS,
    "ppo_friend_dmlpa_transformer": FRIEND_POLICY_KWARGS,
}


def build_model(label: str, vec_env: VecNormalize, seed: int) -> PPO:
    return PPO(
        "MlpPolicy",
        vec_env,
        policy_kwargs=POLICY_KWARGS[label],
        learning_rate=LEARNING_RATE,
        n_steps=N_STEPS,
        batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        clip_range=CLIP_RANGE,
        ent_coef=ENT_COEF,
        vf_coef=VF_COEF,
        max_grad_norm=MAX_GRAD_NORM,
        seed=seed,
        device="auto",
        verbose=1,
    )


def train_policy(label: str, train_seed: int) -> TrainedPolicy:
    policy_dir = output_root / label / f"seed_{train_seed}"
    policy_dir.mkdir(parents=True, exist_ok=True)
    model_path = policy_dir / "model.zip"
    norm_path = policy_dir / "vecnormalize.pkl"
    metadata_path = policy_dir / "metadata.json"

    vec_env = make_vec_env(train_seed)
    model = build_model(label, vec_env, train_seed)
    parameter_count = count_trainable_parameters(model.policy)

    if RESUME_IF_MODEL_EXISTS and model_path.exists() and norm_path.exists():
        print(f"Loading existing model for {label} seed={train_seed}")
        base_vec = vec_env.venv
        vec_env = VecNormalize.load(norm_path, base_vec)
        vec_env.training = False
        vec_env.norm_reward = False
        model = PPO.load(model_path, env=vec_env, device="auto")
        elapsed = 0.0
    else:
        print(f"\n=== Training {label} seed={train_seed} ===")
        print("trainable parameters:", parameter_count)
        started = time.time()
        model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=False)
        elapsed = time.time() - started
        if SAVE_EVERY_TRAINED_MODEL:
            model.save(model_path)
            vec_env.save(norm_path)
            metadata_path.write_text(json.dumps({
                "label": label,
                "train_seed": train_seed,
                "parameter_count": parameter_count,
                "elapsed_seconds": elapsed,
                "model_path": str(model_path),
                "norm_path": str(norm_path),
            }, indent=2))

    return TrainedPolicy(
        label=label,
        train_seed=train_seed,
        model=model,
        vec_env=vec_env,
        model_path=model_path,
        norm_path=norm_path,
        elapsed_seconds=elapsed,
        parameter_count=parameter_count,
    )

In [ ]:
# ==========================
# 8) Train all model/seed runs
# ==========================

trained_policies: list[TrainedPolicy] = []
train_rows = []

for label in POLICY_LABELS:
    for train_seed in TRAIN_SEEDS:
        trained = train_policy(label, train_seed)
        trained_policies.append(trained)
        train_rows.append({
            "policy": trained.label,
            "train_seed": trained.train_seed,
            "parameter_count": trained.parameter_count,
            "elapsed_seconds": trained.elapsed_seconds,
            "model_path": str(trained.model_path),
            "norm_path": str(trained.norm_path),
        })
        pd.DataFrame(train_rows).to_csv(output_root / "training_runs.csv", index=False)

training_df = pd.DataFrame(train_rows)
display(training_df)

In [ ]:
# ========================================================
# 9) Evaluation: fixed protocol, fresh env per episode
# ========================================================

PRIMARY_METRICS = [
    "total_reward",
    "order_level_ret_mean",
    "fill_rate_order_level",
    "backorder_rate_order_level",
    "final_pending_backorders",
    "final_pending_backorder_qty",
    "total_new_delivered",
]


def evaluate_trained_policy(trained: TrainedPolicy) -> pd.DataFrame:
    rows = []
    for episode in range(EVAL_EPISODES):
        eval_seed = 1_000_000 + trained.train_seed * 100 + episode
        eval_env = make_vec_env(eval_seed)
        copy_vecnormalize_stats(trained.vec_env, eval_env)
        obs = eval_env.reset()
        done = np.array([False])
        total_reward = 0.0
        steps = 0
        weekly_rewards = []
        weekly_fill_rates = []
        weekly_ret_cd = []
        total_new_delivered = 0.0
        last_info = {}

        while not bool(done[0]) and steps < MAX_STEPS + 2:
            action, _ = trained.model.predict(obs, deterministic=True)
            obs, reward, done, infos = eval_env.step(action)
            info = dict(infos[0])
            last_info = info
            reward_value = float(reward[0])
            total_reward += reward_value
            if info.get("action_phase") == "weekly_decision":
                steps += 1
                weekly_rewards.append(reward_value)
                if info.get("ret_cd_fill_rate_step") is not None:
                    weekly_fill_rates.append(float(info["ret_cd_fill_rate_step"]))
                if info.get("ret_cd_step") is not None:
                    weekly_ret_cd.append(float(info["ret_cd_step"]))
                total_new_delivered += float(info.get("new_delivered") or 0.0)

        base_env = get_base_env_from_vec(eval_env)
        terminal = get_episode_terminal_metrics(base_env)
        rows.append({
            "policy": trained.label,
            "train_seed": trained.train_seed,
            "episode": episode,
            "eval_seed": eval_seed,
            "steps": steps,
            "total_reward": total_reward,
            "mean_weekly_reward": float(np.mean(weekly_rewards)) if weekly_rewards else np.nan,
            "mean_step_fill_rate": float(np.mean(weekly_fill_rates)) if weekly_fill_rates else np.nan,
            "mean_step_ret_cd": float(np.mean(weekly_ret_cd)) if weekly_ret_cd else np.nan,
            "total_new_delivered": total_new_delivered,
            "final_pending_backorders": float(last_info.get("pending_backorders", np.nan)),
            "final_pending_backorder_qty": float(last_info.get("pending_backorder_qty", np.nan)),
            "order_level_ret_mean": float(terminal.get("order_level_ret_mean", np.nan)),
            "fill_rate_order_level": float(terminal.get("fill_rate_order_level", np.nan)),
            "backorder_rate_order_level": float(terminal.get("backorder_rate_order_level", np.nan)),
        })
        eval_env.close()
    return pd.DataFrame(rows)

all_eval_parts = []
for trained in trained_policies:
    print(f"Evaluating {trained.label} seed={trained.train_seed}")
    part = evaluate_trained_policy(trained)
    all_eval_parts.append(part)
    partial_df = pd.concat(all_eval_parts, ignore_index=True)
    partial_df.to_csv(output_root / "evaluation_episodes.csv", index=False)

eval_df = pd.concat(all_eval_parts, ignore_index=True)
display(eval_df.head())
print("rows:", len(eval_df))

In [ ]:
# =========================================
# 10) Aggregate by train seed and by policy
# =========================================

seed_summary = (
    eval_df
    .groupby(["policy", "train_seed"], as_index=False)[PRIMARY_METRICS]
    .mean()
)
policy_summary = (
    seed_summary
    .groupby("policy", as_index=False)
    .agg({metric: ["mean", "std", "count"] for metric in PRIMARY_METRICS})
)
policy_summary.columns = ["_".join([str(x) for x in col if x]) for col in policy_summary.columns.to_flat_index()]

seed_summary.to_csv(output_root / "seed_summary.csv", index=False)
policy_summary.to_csv(output_root / "policy_summary.csv", index=False)

display(seed_summary)
display(policy_summary)

In [ ]:
# ==============================================================
# 11) Paired comparison, bootstrap CIs, and statistical tests
# ============================================================== 

FRIEND_LABEL = "ppo_friend_dmlpa_transformer"
BASELINE_LABEL = "ppo_mlp_baseline"
BOOTSTRAP_REPS = 20_000
rng = np.random.default_rng(BASE_SEED)


def bootstrap_ci(values: np.ndarray, reps: int = BOOTSTRAP_REPS, alpha: float = 0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    samples = rng.choice(values, size=(reps, len(values)), replace=True).mean(axis=1)
    return float(np.quantile(samples, alpha / 2)), float(np.quantile(samples, 1 - alpha / 2))

comparison_rows = []
for metric in PRIMARY_METRICS:
    pivot = seed_summary.pivot(index="train_seed", columns="policy", values=metric).dropna()
    if FRIEND_LABEL not in pivot.columns or BASELINE_LABEL not in pivot.columns:
        continue
    diff = pivot[FRIEND_LABEL].to_numpy() - pivot[BASELINE_LABEL].to_numpy()
    ci_low, ci_high = bootstrap_ci(diff)
    t_stat, t_p = stats.ttest_rel(pivot[FRIEND_LABEL], pivot[BASELINE_LABEL]) if len(diff) >= 2 else (np.nan, np.nan)
    try:
        w_stat, w_p = stats.wilcoxon(pivot[FRIEND_LABEL], pivot[BASELINE_LABEL], zero_method="wilcox") if len(diff) >= 2 else (np.nan, np.nan)
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    comparison_rows.append({
        "metric": metric,
        "n_paired_train_seeds": int(len(diff)),
        "friend_mean": float(pivot[FRIEND_LABEL].mean()),
        "baseline_mean": float(pivot[BASELINE_LABEL].mean()),
        "paired_diff_friend_minus_baseline": float(np.mean(diff)),
        "bootstrap_ci95_low": ci_low,
        "bootstrap_ci95_high": ci_high,
        "paired_t_p": float(t_p) if np.isfinite(t_p) else np.nan,
        "wilcoxon_p": float(w_p) if np.isfinite(w_p) else np.nan,
        "friend_wins_seed_count": int(np.sum(diff > 0)),
        "baseline_wins_seed_count": int(np.sum(diff < 0)),
        "ties_seed_count": int(np.sum(diff == 0)),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(output_root / "paired_comparison.csv", index=False)
display(comparison_df)

In [ ]:
# ================================
# 12) Figures for paper/debugging
# ================================

fig_dir = output_root / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

plot_metrics = ["total_reward", "order_level_ret_mean", "fill_rate_order_level", "final_pending_backorders"]
for metric in plot_metrics:
    plt.figure(figsize=(7, 4))
    data = [
        seed_summary.loc[seed_summary["policy"] == BASELINE_LABEL, metric].dropna().to_numpy(),
        seed_summary.loc[seed_summary["policy"] == FRIEND_LABEL, metric].dropna().to_numpy(),
    ]
    plt.boxplot(data, labels=["PPO MLP", "Friend DMLPA"], showmeans=True)
    plt.title(metric)
    plt.ylabel(metric)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    path = fig_dir / f"{metric}.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved", path)

# Paired-difference plot for the primary reward metric.
metric = "total_reward"
pivot = seed_summary.pivot(index="train_seed", columns="policy", values=metric).dropna()
if FRIEND_LABEL in pivot.columns and BASELINE_LABEL in pivot.columns:
    diff = pivot[FRIEND_LABEL] - pivot[BASELINE_LABEL]
    plt.figure(figsize=(7, 4))
    plt.axhline(0, color="black", linewidth=1)
    plt.plot(diff.index.astype(str), diff.values, marker="o")
    plt.title("Paired seed differences: Friend DMLPA - PPO MLP")
    plt.ylabel(metric)
    plt.xlabel("train seed")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    path = fig_dir / "paired_diff_total_reward.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved", path)

In [ ]:
# =========================================
# 13) Machine-readable conclusion template
# =========================================

primary_metric = "total_reward"
primary_row = comparison_df.loc[comparison_df["metric"] == primary_metric].iloc[0].to_dict()
if primary_row["bootstrap_ci95_low"] > 0:
    conclusion = "Friend DMLPA outperformed PPO MLP on the primary reward metric under this protocol."
elif primary_row["bootstrap_ci95_high"] < 0:
    conclusion = "PPO MLP outperformed Friend DMLPA on the primary reward metric under this protocol."
else:
    conclusion = "The primary reward difference is inconclusive under this protocol."

result_card = {
    "primary_metric": primary_metric,
    "conclusion": conclusion,
    "primary_comparison": primary_row,
    "output_root": str(output_root),
    "claim_boundary": (
        "This is a PPO architecture comparison under the thesis-faithful DKANA "
        "Gymnasium control environment. It is not a claim that RL reproduces "
        "the original thesis exactly, and it is not a Track B downstream-control claim."
    ),
}
(output_root / "result_card.json").write_text(json.dumps(result_card, indent=2))
print(json.dumps(result_card, indent=2))

In [ ]:
# ======================
# 14) Zip final artifacts
# ======================

archive_base = output_root.parent / f"{output_root.name}_artifacts"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=output_root)
print("artifact zip:", archive_path)
print("files written:")
for path in sorted(output_root.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(output_root), f"({path.stat().st_size / 1024:.1f} KiB)")

## Claim discipline for the paper

Use this notebook to support a narrow architecture claim only after the serious profile completes. Safe wording:

> Under the same thesis-faithful DKANA control environment, reward, risk level, frame stack, training budget, and evaluation protocol, we compared a standard PPO MLP policy with a PPO policy using a DMLPA-style Transformer feature extractor.

Do not claim that this proves Track B, downstream dispatch control, or exact thesis reproduction. This notebook tests **architecture under the thesis-faithful DKANA Gym interface**.